## Stroke Work
<br>Author: Daniel Maina Nderitu<br>
Project: MADIVA<br>
Purpose: Survival logic (VERY IMPORTANT)<br>
Notes:   How stroke incidence was constructed<br>
Determining how long an individual was under observation and whether they developed stroke during that time

#### Bootstrap cell

In [1]:
# =================== BOOTSTRAP CELL ===================
# Standard setup for all notebooks
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[0]  # assumes notebooks are in a subfolder
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# ========================================================
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.config.variables import COVARIATES

# ========================================================
# For warnings and nicer plots
import warnings
warnings.filterwarnings("ignore")
sns.set(style="whitegrid")

import sys
from pathlib import Path

# ========================================================
# Ensure project root is in Python path
# Adjust this if your notebooks are nested deeper
PROJECT_ROOT = Path.cwd().parents[0]  # assumes notebooks are in a subfolder
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# ========================================================
# Import helper to load paths
from src.utils.helpers import load_paths

# ========================================================
# Load paths from config.yaml (works regardless of notebook location)
paths = load_paths()

# ========================================================
# Print paths to confirm
for key, value in paths.items():
    print(f"{key}: {value}")

# ========================================================
# Using these paths in my notebook:
DATA_DIR = paths['DATA_DIR']
OUT_DIR = paths['OUT_DIR']
FIG_DIR = paths['FIG_DIR']

# ========================================================

BASE_DIR: D:\APHRC\GoogleDrive_ii\stata_do_files\madiva\stroke_work
DATA_DIR: D:\APHRC\GoogleDrive_ii\stata_do_files\madiva\stroke_work\data
OUT_DIR: D:\APHRC\GoogleDrive_ii\stata_do_files\madiva\stroke_work\model_output
FIG_DIR: D:\APHRC\GoogleDrive_ii\stata_do_files\madiva\stroke_work\visualization
MODEL_DIR: D:\APHRC\GoogleDrive_ii\stata_do_files\madiva\stroke_work\model_output\statsmodels
NOTEBOOKS_DIR: D:\APHRC\GoogleDrive_ii\stata_do_files\madiva\stroke_work\notebooks
NOTEBOOKS_EXECUTED_DIR: D:\APHRC\GoogleDrive_ii\stata_do_files\madiva\stroke_work\notebooks_executed


### Import data - from previous step

In [2]:
# data saved as pickle:
df = pd.read_pickle(OUT_DIR / "df_step03_processed.pkl")

### Event, time_at_risk (person-years), covariates

Treatment to ensure we don't inflate incidence

In [3]:
print(df['stroke_status_derived'].value_counts()) 

stroke_status_derived
0      28149
1       1491
999      506
Name: count, dtype: int64


In [4]:
df.head()

,individual_id,age,sex,hdss_name,alco_ever,alco_12m,alco_30d,alco_bing_y,tobac_ever,tobac_cur,...,source_Diabetics_Baseline,source_Diabetics_Followup,source_HAALSI_1,source_HAALSI_2,source_HAALSI_3,source_HIV_NCD,source_Nkateko_1,source_Nkateko_2,source_SCALEUP_Clinic_Baseline,source_SCALEUP_Pop_Baseline
0,BBBHY,33,2,Agincourt,NaN,NaN,NaN,NaN,0.0,888.0,...,False,False,False,False,False,False,False,False,False,False
1,BBBHY,34,2,Agincourt,NaN,NaN,NaN,NaN,999.0,NaN,...,False,False,False,False,False,False,False,False,False,False
2,BBBNE,46,1,Agincourt,1.0,1.0,1.0,NaN,1.0,0.0,...,False,False,False,False,False,True,False,False,False,False
3,BBBNE,49,1,Agincourt,NaN,NaN,NaN,NaN,1.0,0.0,...,False,False,False,False,False,False,True,False,False,False
4,BBBNE,50,1,Agincourt,1.0,NaN,1.0,1.0,1.0,0.0,...,False,False,True,False,False,False,False,False,False,False


#### obs_date Parsing, Study Periods, Start date and study end

In [5]:
df.head()

,individual_id,age,sex,hdss_name,alco_ever,alco_12m,alco_30d,alco_bing_y,tobac_ever,tobac_cur,...,source_Diabetics_Baseline,source_Diabetics_Followup,source_HAALSI_1,source_HAALSI_2,source_HAALSI_3,source_HIV_NCD,source_Nkateko_1,source_Nkateko_2,source_SCALEUP_Clinic_Baseline,source_SCALEUP_Pop_Baseline
0,BBBHY,33,2,Agincourt,NaN,NaN,NaN,NaN,0.0,888.0,...,False,False,False,False,False,False,False,False,False,False
1,BBBHY,34,2,Agincourt,NaN,NaN,NaN,NaN,999.0,NaN,...,False,False,False,False,False,False,False,False,False,False
2,BBBNE,46,1,Agincourt,1.0,1.0,1.0,NaN,1.0,0.0,...,False,False,False,False,False,True,False,False,False,False
3,BBBNE,49,1,Agincourt,NaN,NaN,NaN,NaN,1.0,0.0,...,False,False,False,False,False,False,True,False,False,False
4,BBBNE,50,1,Agincourt,1.0,NaN,1.0,1.0,1.0,0.0,...,False,False,True,False,False,False,False,False,False,False


#### time_at_risk, offset

In [6]:
# ***************************************************************************
# ******************************** start ************************************
# ***************************************************************************
# 2. Ordering observations (arranging data chronologically) <<<<<<<<<<<<<<<<<
# ---------------------------------------------------------------------------
df = df.sort_values(['individual_id', 'obs_date'])

# ***************************************************************************
# ******************************** start ************************************
# ***************************************************************************
# 3. Defining observation window (risk continues until study end) <<<<<<<<<<<
#    A person is at risk from visit date until study end
# ---------------------------------------------------------------------------
df['start_date'] = df['obs_date']
df['end_date']   = df['study_end']

# ---------------------------------------------------------------------------

# ***************************************************************************
# ******************************** start ************************************
# ***************************************************************************
# 4. Computing time at risk (person-time) in months (can be done in years)
#    Incidence depends on exposure *****
#    Handling missing time through imputation (assuming typical follow-up when unknown) *****
# ---------------------------------------------------------------------------
df['time_at_risk'] = (df['end_date'] - df['start_date']).dt.days / 30.4375 # 365.25

missing_time_count = df['time_at_risk'].isna().sum()
total_count = len(df)
missing_percent = (missing_time_count / total_count) * 100
print(f"Missing time_at_risk before imputation: {missing_time_count:,} ({missing_percent:.2f}%)")
# ---------------------------------------------------------------------------
# Using the site variable to examine missingness by site
# ---------------------------------------------------------------------------
if 'site' in df.columns:
    missing_by_site = df.groupby('hdss_name')['time_at_risk'].apply(lambda x: x.isna().mean() * 100).round(2)
    print("\nMissing time_at_risk by site (%):")
    print(missing_by_site)

df['time_at_risk'] = df['time_at_risk'].fillna(df['time_at_risk'].median())  
# df['time_at_risk'] = df['time_at_risk'].fillna(1.5) 
# ---------------------------------------------------------------------------

# ***************************************************************************
# ******************************** start ************************************
# ***************************************************************************
# 5. Avoiding zero follow-up <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
#    Imposing a tiny minimum risk time
# ---------------------------------------------------------------------------
df.loc[df['time_at_risk'] < 0.01, 'time_at_risk'] = 0.01  
# avoid log(0)  [Doing some house cleaning to avoid negative offset values]

# ***************************************************************************
# ******************************** start ************************************
# ***************************************************************************
# 6. Creating offset variable  <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
#    (Added a tiny constant (epsilon) since I was encountering some negative offset values. 
#    A robust and common approach in survival and Poisson models.)
#    In poison models ------   log(expected strokes) = predictors + log(person-time)
# ---------------------------------------------------------------------------
df['offset'] = np.log(df['time_at_risk']) #.clip(lower=1e-6)) 

# ***************************************************************************
# ******************************** start ************************************
# ***************************************************************************
# 7. Defining stroke event (stroke occurrence) <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
#              Identifying new stroke events (Marks when new stroke happens)    
# ---------------------------------------------------------------------------
# Individuals with previous stroke and current no-stroke
df['stroke_corrected'] = (
    df.groupby('individual_id')['stroke_status_derived']
      .cummax()
)

df['stroke_prev'] = (
    df.groupby('individual_id')['stroke_status_derived']
        .shift(1)     # Looking at previous observation
        .fillna(0)    # handles the first record for each person (where there is no previous data)
) 
                
                
df['event'] = np.where(
    (df['stroke_prev'] == 0) & 
    (df['stroke_status_derived'] == 1),   # Identifying incident stroke event
    1, 0)
df['event'] = df['event'].fillna(0)
print("Debugging  ************* - df['stroke_prev']", df['stroke_prev'].value_counts(), "df['stroke_status_derived']",  df['stroke_status_derived'].value_counts() )

# ***************************************************************************
# Debugging purposes - step 1
# summary
columns_to_check = ['stroke_prev', 'stroke_status_derived', 'event'] #, 'cumulative_stroke'
# For each column, get value counts
print("=== Debugging purposes - step 1 ===")
print("=== stroke_prev ===")
print(df['stroke_prev'].value_counts())
print("\n")

print("=== stroke_status_derived ===")
print(df['stroke_status_derived'].value_counts())
print("\n")

print("=== event ===")
print(df['event'].value_counts())
print("\n")

# ***************************************************************************
# ****************************** 8. start ***********************************
# ***************************************************************************
# 8. Finding the first event per person  (different from prevalence)
#    time-to-first-event dataset
df['cumulative_stroke'] = df.groupby('individual_id')['stroke_status_derived'].cumsum()
# keeping only records before or up to the first stroke (dropping all post-stroke observations)
df = df[df['cumulative_stroke'] <= 1]

print("============ Confirming that each person has ≤1 event ============")
print("Events per person",df.groupby('individual_id')['event'].sum().value_counts())

# ***************************************************************************
# Debugging purposes - step 2
# summary
columns_to_check = ['stroke_prev', 'stroke_status_derived', 'event'] #, 'cumulative_stroke'
# For each column, get value counts
print("=== Debugging purposes - step 2 ===")
print("=== stroke_prev ===")
print(df['stroke_prev'].value_counts())
print("\n")

print("=== stroke_status_derived ===")
print(df['stroke_status_derived'].value_counts())
print("\n")

print("=== event ===")
print(df['event'].value_counts())
print("\n")


# ---------------------------------------------------------------------------
# ---------------------------------------------------------------------------

print("After filtering to <= first stroke, rows:", df.shape[0])

# ***************************************************************************
# ******************************** start ************************************
# ***************************************************************************
# How many times is each person represented?
# ---------------------------
counts = df['individual_id'].value_counts()          # number of rows per person (index = individual_id)
counts_summary = counts.describe()                   # mean, min, max, median, etc.

print("\nRecords per individual (summary):")
print(counts_summary)

# How many individuals have only one record (single-visit)?
n_single = (counts == 1).sum()
pct_single = (n_single / counts.shape[0]) * 100

print(f"\nIndividuals with a single record: {n_single} ({pct_single:.1f}%)")

# ***************************************************************************
# ******************************** start ************************************
# ***************************************************************************
# Show frequency distribution (top few)
print("\nTop frequency counts (number of Agincourt persons with X records):")
freq_table = counts.value_counts().sort_index()      # index = number of records, value = # persons
print(freq_table.head(20))  # show first 20 rows; increase if needed

# # The full distribution dataframe:
# freq_df = freq_table.reset_index().rename(columns={'index': 'n_records', 'individual_id': 'n_persons'})
# Create a clean dataframe version
freq_df = freq_table.reset_index(name="n_persons")
freq_df.columns = ['n_records', 'n_persons']  # rename safely

Missing time_at_risk before imputation: 0 (0.00%)

Missing time_at_risk by site (%):
hdss_name
Agincourt    0.0
Nairobi      0.0
Name: time_at_risk, dtype: float64
Debugging  ************* - df['stroke_prev'] stroke_prev
0.0      29145
1.0        842
999.0      159
Name: count, dtype: int64 df['stroke_status_derived'] stroke_status_derived
0      28149
1       1491
999      506
Name: count, dtype: int64
=== Debugging purposes - step 1 ===
=== stroke_prev ===
stroke_prev
0.0      29145
1.0        842
999.0      159
Name: count, dtype: int64


=== stroke_status_derived ===
stroke_status_derived
0      28149
1       1491
999      506
Name: count, dtype: int64


=== event ===
event
0    29473
1      673
Name: count, dtype: int64


============ Confirming that each person has ≤1 event ============
Events per person event
0    8852
1     673
Name: count, dtype: int64
=== Debugging purposes - step 2 ===
=== stroke_prev ===
stroke_prev
0.0    28822
Name: count, dtype: int64


=== stroke_status

#### Interraction terms

In [7]:
df["hpt_diab_interaction"] = df["hpt_status_derived"] * df["diab_status_derived"]

df["hpt_site_interaction"] = df["hpt_status_derived"] * df["site_Nairobi"]
df["site_diab_interaction"] = df["site_Nairobi"] * df["diab_status_derived"]
df["bmi_sex_interaction"] = df["bmi_category_Overweight_Obese"] * df["sex_binary"]

# site × hypertension, site × diabetes, or BMI × sex

#### Checking

In [8]:
print(df.shape)
print(df['stroke_prev'].value_counts())
print(df['stroke_status_derived'].value_counts()) 
print(df['event'].value_counts())
print(df['cumulative_stroke'])
df['cumulative_stroke'].value_counts()

(28822, 366)
stroke_prev
0.0    28822
Name: count, dtype: int64
stroke_status_derived
0    28149
1      673
Name: count, dtype: int64
event
0    28149
1      673
Name: count, dtype: int64
24278    0
24279    0
24280    0
24281    0
24282    0
        ..
24273    0
24274    0
24275    0
24276    0
24277    0
Name: cumulative_stroke, Length: 28822, dtype: int64


cumulative_stroke
0    28149
1      673
Name: count, dtype: int64

In [9]:
df[['offset', 'time_at_risk']]

,offset,time_at_risk
24278,2.093713,8.114990
24279,0.966351,2.628337
24280,2.395466,10.973306
24281,1.346499,3.843943
24282,-2.029381,0.131417
...,...,...
24273,2.175312,8.804928
24274,2.578286,13.174538
24275,1.708289,5.519507
24276,2.365068,10.644764


#### Stroke transition

In [10]:
# Identify individuals who experienced a stroke (event == 1)
stroke_cases = df[df['event'] == 1]['individual_id'].unique()

# Subset full data for only those individuals
df_stroke_transition = df[df['individual_id'].isin(stroke_cases)]
selected = ['individual_id', 'start_date', 'end_date', 'stroke_prev', 'event'
            , 'time_at_risk', 'cumulative_stroke', 'offset'] # , 'res_eventdate'

check_df = df_stroke_transition[selected].sort_values(['individual_id', 'start_date']).head(20)
check_df.describe()

,start_date,end_date,stroke_prev,event,time_at_risk,cumulative_stroke,offset
count,20,20,20.0,20.000000,20.000000,20.000000,20.000000
mean,2012-08-14 12:00:00,2013-01-11 01:12:00,0.0,0.500000,4.913347,0.500000,1.433005
min,2008-07-22 00:00:00,2009-03-30 00:00:00,0.0,0.000000,2.234086,0.000000,0.803832
25%,2012-08-10 00:00:00,2012-12-19 00:00:00,0.0,0.000000,2.611910,0.000000,0.959533
50%,2012-09-29 00:00:00,2012-12-19 00:00:00,0.0,0.500000,4.106776,0.500000,1.412606
75%,2013-06-02 12:00:00,2014-10-09 18:00:00,0.0,1.000000,4.854209,1.000000,1.573802
max,2015-08-06 00:00:00,2015-11-30 00:00:00,0.0,1.000000,19.778234,1.000000,2.984582
std,NaN,NaN,0.0,0.512989,3.826270,0.512989,0.516849


#### End

In [11]:
# Saved as pickle (faster for large data, preserves types)
df.to_pickle(OUT_DIR / "df_step04_processed.pkl")